In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json ,to_json,col,struct
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType


In [2]:
spark = SparkSession.builder \
 .appName("FraudDetection") \
 .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
 .getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5450b51c-d70b-4996-a9ad-7e11baac2152;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1123ms :: artifacts dl 43ms
	:: modules in u

In [7]:
spark.sparkContext.setLogLevel("WARN")


In [8]:
user_df = spark.read.csv("data/user_table.csv", header=True ,inferSchema= True)



In [9]:
tx_schema = StructType([
     StructField("tx_id", IntegerType(), True),
     StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),

     StructField("timestamp", DoubleType(), True)

] )


In [10]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()


In [11]:
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")

fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [12]:
enriched_fraud = fraud_stream.join(user_df, "userId")


In [13]:
output_stream = enriched_fraud \
     .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")


In [14]:

fraud_stream2 = parsed_stream.filter(col("amount") > 5000.0)

In [15]:
enriched_fraud2 = fraud_stream2.join(user_df, "userId")

In [16]:
output_stream2 = enriched_fraud2\
     .withColumn("value", to_json(struct("*")).cast("string")) \
     .select("value")

In [ ]:
query = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification") \
 .option("checkpointLocation", "/workspace/checkpoints/query") \
 .start()

query2 = output_stream2.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification2") \
 .option("checkpointLocation", "/workspace/checkpoints/query2") \
 .start()
query2.awaitTermination()
query.awaitTermination()



26/06/19 06:18:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:18:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:18:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 06:18:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                